# Hamilton-Jacobi Reachability with Spatially-Varying Disturbance Landscapes
This notebook implements backward reachability analysis for a double integrator system subjected to a state-dependent adversarial disturbance bound field $d \in [-D(x), D(x)]$. 

We evaluate and compare two formulations:
1. **Full Game-Theoretic Formulation:** The controller fights a worst-case spatial adversary.
2. **Net Capability Formulation (Underapproximation):** The disturbance channel is zeroed out, but the control authority is conservatively scaled down point-by-point to $U_{net}(x) = U_{max} - D(x)$.

We benchmark these formulations across 3 distinct environmental disturbance profiles.

In [ ]:
# Imports

import jax
import jax.numpy as jnp
import numpy as np
import time
from tqdm.notebook import tqdm

from IPython.display import HTML
import matplotlib.animation as anim
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.graph_objects as go

import hj_reachability as hj

from hj_reachability.systems.doubleint import DoubleIntegrator

In [ ]:
# Implement Custom Dynamics Subclasses strictly adhering to dynamics.py signatures

class DoubleIntDynamics(hj.Dynamics):
    """True game-theoretic system where control minimizes and disturbance maximizes."""
    def __init__(self, u_max, landscape_fn):
        self.u_max = u_max
        self.landscape_fn = landscape_fn
        # Initialize base abstract class with tracking modes
        super().__init__(
            control_mode="min", 
            disturbance_mode="max", 
            control_space=None, 
            disturbance_space=None
        )

    def __call__(self, state, control, disturbance, time):
        x2 = state[1]
        # Extract inputs handling potential array-wrapping from the level-set solver
        u = control[0] if hasattr(control, "__len__") else control
        d = disturbance[0] if hasattr(disturbance, "__len__") else disturbance
        return jnp.array([x2, u + d])

    def optimal_control_and_disturbance(self, state, time, grad_value):
        D_x = self.landscape_fn(state)
        
        # control_mode="min" minimizes the Hamiltonian -> opposite sign of velocity gradient
        u_opt = -jnp.sign(grad_value[1]) * self.u_max
        
        # disturbance_mode="max" maximizes the Hamiltonian -> tracks sign of velocity gradient
        d_opt = jnp.sign(grad_value[1]) * D_x
        
        return jnp.array([u_opt]), jnp.array([d_opt])

    def partial_max_magnitudes(self, state, time, value, grad_value_box):
        D_x = self.landscape_fn(state)
        # Dimension 0 (Position): dx1/dt = x2 -> max speed is |x2|
        # Dimension 1 (Velocity): dx2/dt = u + d -> worst-case combined speed is U_max + D(x)
        return jnp.array([jnp.abs(state[1]), self.u_max + D_x])


class NetCapabilityDynamics(hj.Dynamics):
    """Conservative underapproximation system: zero disturbance, shrunk control authority."""
    def __init__(self, u_max, landscape_fn):
        self.u_max = u_max
        self.landscape_fn = landscape_fn
        super().__init__(
            control_mode="min", 
            disturbance_mode="max", 
            control_space=None, 
            disturbance_space=None
        )

    def __call__(self, state, control, disturbance, time):
        x2 = state[1]
        u = control[0] if hasattr(control, "__len__") else control
        return jnp.array([x2, u])

    def optimal_control_and_disturbance(self, state, time, grad_value):
        D_x = self.landscape_fn(state)
        u_net = jnp.maximum(0.0, self.u_max - D_x)  # Clamp lower bound to physical zero
        
        u_opt = -jnp.sign(grad_value[1]) * u_net
        return jnp.array([u_opt]), jnp.array([0.0])

    def partial_max_magnitudes(self, state, time, value, grad_value_box):
        D_x = self.landscape_fn(state)
        u_net = jnp.maximum(0.0, self.u_max - D_x)
        # u_net = self.u_max - D_x  # Allow negative values to reflect potential backward motion
        return jnp.array([jnp.abs(state[1]), u_net])

In [ ]:
# Disturbance Landscapes

def landscape_1_uniform(state, d_max=1):
    """1. Uniform baseline disturbance everywhere."""
    return d_max

def landscape_2_patterned(state, d_base=0.8, A=0.25, wavelength=4.0):
    """2. Cyclical patterned disturbance using a sinusoidal wave."""
    x1 = state[0]
    return d_base + A * jnp.cos(2 * jnp.pi * x1 / wavelength)

def landscape_3_spike(state, d_base=0.1, A=0.55, center=0.0, sigma=0.35):
    """3. Highly localized wind corridor modeled via a Gaussian spike."""
    x1 = state[0]
    return d_base + A * jnp.exp(-((x1 - center) ** 2) / (2 * sigma ** 2))

def landscape_4_radial_growth(state, d_base=0.5, L=0.25):
    """4. Disturbance magnitude increasing linearly from the origin with Lipschitz constant L."""
    # Euclidean distance from origin: r = ||x||_2 = sqrt(x1^2 + x2^2)
    r = jnp.linalg.norm(state)
    return d_base + L * r

In [ ]:
# Grid Generation and Target Set Initialization

grid_lower = jnp.array([-10.0, -10.0])
grid_upper = jnp.array([10.0, 10.0])
grid_shape = (201, 201)

# Expand the grid bounds so the solver can trace the full trajectory expansion
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    domain=hj.sets.Box(
        grid_lower,     # Lower bounds: [min_position, min_velocity]
        grid_upper      # Upper bounds: [max_position, max_velocity]
    ),
    shape=grid_shape             # Grid resolution
)

# grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
#     hj.sets.Box(grid_lower, grid_upper), grid_shape
# )

# Safe target boundary defined as a terminal circle centered at the origin
safety_bound = 5
initial_values = jnp.linalg.norm(grid.states, axis=-1) - safety_bound

print(f"Grid successfully instantiated with shape {grid_shape}.")

In [ ]:
# Random Lipschitz-Continuous Ground Truth Disturbance Generator

def create_random_lipschitz_landscape(
    grid, 
    L_D=0.25, 
    d_base=0.1, 
    num_centers=12, 
    sigma=2.5, 
    seed=101
):
    """5. Random Lipschitz-Continuous Ground Truth Disturbance Generator."""
    key = jax.random.PRNGKey(seed)
    key_c, key_w = jax.random.split(key)

    # Sample random center locations c_k within grid bounds and random weights w_k in [-1, 1]
    grid_low = grid.domain.lo
    grid_high = grid.domain.hi
    centers = jax.random.uniform(
        key_c, shape=(num_centers, 2), 
        minval=grid_low, maxval=grid_high
    )
    weights = jax.random.uniform(
        key_w, shape=(num_centers,), 
        minval=-1.0, maxval=1.0
    )

    # Unscaled RBF function sum for a single state (2,)
    def unscaled_f(state):
        diffs = state - centers  # Shape: (num_centers, 2)
        dist_sq = jnp.sum(diffs**2, axis=-1)
        return jnp.sum(weights * jnp.exp(-dist_sq / (2.0 * sigma**2)))

    # Measure spatial gradient ||grad f(x)||_2 across the grid using vmap
    grad_fn = jax.grad(unscaled_f)
    grid_states = grid.states  # Shape: (201, 201, 2)
    
    # Vectorize unscaled_f and grad_fn over the 2D grid
    unscaled_map = jax.vmap(jax.vmap(unscaled_f))(grid_states)         # Shape: (201, 201)
    spatial_grads = jax.vmap(jax.vmap(grad_fn))(grid_states)          # Shape: (201, 201, 2)
    grad_norms = jnp.linalg.norm(spatial_grads, axis=-1)              # Shape: (201, 201)
    
    max_measured_grad = jnp.max(grad_norms)
    min_unscaled_val = jnp.min(unscaled_map)

    # Scale factor so peak spatial gradient equals L_D exactly
    scale_factor = L_D / max_measured_grad

    # Final Ground Truth Disturbance Function D_true(state)
    def d_true_fn(state):
        raw_val = scale_factor * unscaled_f(state)
        # Shift baseline so minimum value across grid is d_base
        return raw_val - scale_factor * min_unscaled_val + d_base

    
    return d_true_fn

In [ ]:
# Parameters for the random Lipschitz disturbance landscape

L_D_target = 0.25   # Desired Lipschitz constant
d_base = 0.1        # Minimum disturbance magnitude
seed = 102          # Random seed

In [ ]:
# Verification & Inspection Plot for random Lipschitz-Continuous Ground Truth Disturbance

# Generate Ground Truth Landscape
d_true_fn_verify = create_random_lipschitz_landscape(
    grid, L_D=L_D_target, d_base=d_base, seed=seed
)

# Evaluate D_true over grid
d_true_map = jax.vmap(jax.vmap(d_true_fn_verify))(grid.states)

# Verify local gradient magnitudes ||grad D_true(x)||_2
grad_d_fn = jax.grad(d_true_fn_verify)
grad_map = jax.vmap(jax.vmap(grad_d_fn))(grid.states)
grad_norm_map = jnp.linalg.norm(grad_map, axis=-1)

print("==================================================")
print(f"Target Lipschitz Constant (L_D)  : {L_D_target:.4f}")
print(f"Max Measured Spatial Gradient   : {jnp.max(grad_norm_map):.4f}")
print(f"Min Disturbance Value           : {jnp.min(d_true_map):.4f}")
print(f"Max Disturbance Value           : {jnp.max(d_true_map):.4f}")
print("==================================================")

# Plot Ground Truth Landscape and Gradient Norm Map
x1_coords = grid.coordinate_vectors[0]
x2_coords = grid.coordinate_vectors[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: D_true(x)
cf1 = axes[0].contourf(x1_coords, x2_coords, d_true_map.T, levels=20, cmap='viridis')
fig.colorbar(cf1, ax=axes[0], label="Disturbance Magnitude $D_{true}(x)$")
axes[0].set_title("Ground Truth Disturbance Landscape $D_{true}(x)$", fontweight="bold")
axes[0].set_xlabel("Position ($x_1$)")
axes[0].set_ylabel("Velocity ($x_2$)")
axes[0].grid(True, linestyle=":", alpha=0.5)

# Plot 2: ||grad D_true(x)||_2 to verify Lipschitz bound
nabla = "\u2207"
cf2 = axes[1].contourf(x1_coords, x2_coords, grad_norm_map.T, levels=20, cmap='magma')
cbar2 = fig.colorbar(cf2, ax=axes[1], label="Gradient Norm $| " + nabla + " D_{true}(x) |_2$")
cbar2.ax.axhline(L_D_target, color='red', linestyle='--', linewidth=2, label="Max $L_D$ Bound")
axes[1].set_title("Spatial Gradient Norm $| " + nabla + " D_{true}(x) |_2 <= L_D$", fontweight="bold")
axes[1].set_xlabel("Position ($x_1$)")
axes[1].grid(True, linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# PLOT POINTS AND INDIVIDUAL CONES FOR VISUALIZATION

u_max = 1.0   # Control Authority

d_true_fn = create_random_lipschitz_landscape(
    grid, L_D=L_D_target, d_base=d_base, seed=102
)

sampled_points = jnp.array([
    [-5.0,  3.0],
    [-5.5, 3.5],
    [-3.0, -4.0],
    # [ 1.0, -5.0]
])

sampled_disturbances = jax.vmap(d_true_fn)(sampled_points)

def create_single_cone_fn(x_sample, d_sample, L_D_bound):
    def cone_fn(state):
        dist = jnp.linalg.norm(state - x_sample)
        return d_sample + L_D_bound * dist
    return cone_fn

cone_functions = [
    create_single_cone_fn(sampled_points[i], sampled_disturbances[i], L_D_target)
    for i in range(len(sampled_points))
]

d_true_map = jax.vmap(jax.vmap(d_true_fn))(grid.states)
cone_maps = [jax.vmap(jax.vmap(c_fn))(grid.states) for c_fn in cone_functions]

# Color scale strictly bounded by control authority (vmax = u_max)
vmin = float(np.floor(d_true_map.min() * 10) / 10.0)  # e.g., 0.10
vmax = u_max                                          # e.g., 1.00

# Color levels strictly within the Controllable Trust Region [0.1, 1.0]
levels = np.linspace(vmin, vmax, 20)


# VISUALIZATION: Ground Truth vs. Individual Cones

x1_coords = grid.coordinate_vectors[0]
x2_coords = grid.coordinate_vectors[1]

num_plots = 1 + len(cone_maps)  # Total number of subplots: 1 for ground truth + 4 cones
fig_width = 26.5/5 * num_plots

fig, axes = plt.subplots(1, num_plots, figsize=(fig_width, 6.0), sharey=True)

for ax in axes:
    ax.set_aspect('equal')

# Plot 0: Ground Truth D_true(x)
cf0 = axes[0].contourf(x1_coords, x2_coords, d_true_map.T, levels=levels, cmap='viridis', extend='max')
axes[0].scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', marker='*', s=150, zorder=5)
axes[0].set_title("Ground Truth $D_{true}(x)$", fontweight="bold", fontsize=11)
axes[0].set_xlabel("Position ($x$)")
axes[0].set_ylabel("Velocity ($\\dot{x}$)")
axes[0].grid(True, linestyle=":", alpha=0.5)

# Plots 1-4: Individual Cones showing Controllable Trust Circles
for i, cone_map in enumerate(cone_maps):
    ax = axes[i + 1]
    
    # Fill contours ONLY inside the controllable region (D <= u_max)
    cf = ax.contourf(x1_coords, x2_coords, cone_map.T, levels=levels, cmap='viridis')
    
    # Draw red dotted Trust Boundary (D = u_max)
    ax.contour(x1_coords, x2_coords, cone_map.T, levels=[u_max], colors='red', linestyles='dotted', linewidths=2.0)
    
    # Draw sample marker
    ax.scatter(sampled_points[i, 0], sampled_points[i, 1], color='red', marker='*', s=150, zorder=5)
    
    # Calculate analytical trust radius for title
    d_val = float(sampled_disturbances[i])
    r_trust = max(0.0, (u_max - d_val) / L_D_target) if d_val <= u_max else 0.0
    
    ax.set_title(f"Cone {i+1} ($d_{{i}}={d_val:.2f}, r_{{trust}}={r_trust:.2f}$)", fontweight="bold", fontsize=10)
    ax.set_xlabel("Position ($x$)")
    ax.grid(True, linestyle=":", alpha=0.5)

# Colorbar
cbar_ax = fig.add_axes([0.92, 0.18, 0.01, 0.68])
cbar = fig.colorbar(cf0, cax=cbar_ax, format='%.2f')
cbar.set_label(f"Controllable Disturbance Scale $D(x) ≤ u_{{max}} = {u_max}$", labelpad=15)

plt.subplots_adjust(left=0.04, right=0.91, bottom=0.18, top=0.88, wspace=0.02)
plt.show()

In [ ]:
# PLOT COMBINED LANDSCAPE (POINTWISE MINIMUM OF ALL CONES)

def combined_disturbance_fn(state):
    """
    Computes D_combined(x) = min_i ( d_i + L_D_target * ||x - x_i||_2 )
    takes the minimum overapproximation of disturbance at point i among all sampled cones.
    """
    dists = jnp.linalg.norm(state - sampled_points, axis=-1)  # Shape: (K,)
    cone_values = sampled_disturbances + L_D_target * dists         # Shape: (K,)
    return jnp.min(cone_values)                              # Scalar min

# Evaluate maps over grid
d_true_map = jax.vmap(jax.vmap(d_true_fn))(grid.states)
d_combined_map = jax.vmap(jax.vmap(combined_disturbance_fn))(grid.states)


# VISUALIZATION: Ground Truth vs. Combined Landscape

vmin = float(np.floor(d_true_map.min() * 10) / 10.0)
vmax = u_max
levels = np.linspace(vmin, vmax, 20)

x1_coords = grid.coordinate_vectors[0]
x2_coords = grid.coordinate_vectors[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6.0), sharey=False)

for ax in axes:
    ax.set_aspect('equal')

# --- Plot 1: Ground Truth D_true(x) ---
cf0 = axes[0].contourf(x1_coords, x2_coords, d_true_map.T, levels=levels, cmap='viridis', extend='max')
axes[0].scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', marker='*', s=150, zorder=5)

axes[0].set_title("Ground Truth $D_{true}(x)$", fontweight="bold", fontsize=12)
axes[0].set_xlabel("Position ($x$)")
axes[0].set_ylabel("Velocity ($\\dot{x}$)")
axes[0].grid(True, linestyle=":", alpha=0.5)

# --- Plot 2: Combined Landscape D_combined(x) ---
cf1 = axes[1].contourf(x1_coords, x2_coords, d_combined_map.T, levels=levels, cmap='viridis')
axes[1].contour(x1_coords, x2_coords, d_combined_map.T, levels=[u_max], colors='red', linestyles='dotted', linewidths=2.0)
axes[1].scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', marker='*', s=150, zorder=5)

axes[1].set_title("Combined Overapproximation $\\bar{D}_{combined}(x) = \\min_i \\bar{D}_i(x)$", fontweight="bold", fontsize=12)
axes[1].set_xlabel("Position ($x$)")
axes[1].set_ylabel("Velocity ($\\dot{x}$)")
axes[1].grid(True, linestyle=":", alpha=0.5)

# --- Colorbar ---
cbar_ax = fig.add_axes([0.92, 0.18, 0.015, 0.68])
cbar = fig.colorbar(cf0, cax=cbar_ax, format='%.2f')
cbar.set_label(f"Controllable Disturbance Scale $D(x) ≤ u_{{max}} = {u_max}$", labelpad=15, fontsize=11)

plt.subplots_adjust(left=0.08, right=0.90, bottom=0.15, top=0.88, wspace=0.04)
plt.show()

In [ ]:
# PLOT POINTS AND INDIVIDUAL CONES FOR VISUALIZATION

u_max = 1.0   # Control Authority

d_true_fn = create_random_lipschitz_landscape(
    grid, L_D=L_D_target, d_base=d_base, seed=102
)

sampled_points = jnp.array([
    [-5.0,  3.0],
    [-5.5, 3.5],
    [-3.0, -4.0],
    # [ 1.0, -5.0]
])

sampled_disturbances = jax.vmap(d_true_fn)(sampled_points)

def create_single_cone_fn(x_sample, d_sample, L_D_bound):
    def cone_fn(state):
        dist = jnp.linalg.norm(state - x_sample)
        return d_sample + L_D_bound * dist
    return cone_fn

cone_functions = [
    create_single_cone_fn(sampled_points[i], sampled_disturbances[i], L_D_target)
    for i in range(len(sampled_points))
]

d_true_map = jax.vmap(jax.vmap(d_true_fn))(grid.states)
cone_maps = [jax.vmap(jax.vmap(c_fn))(grid.states) for c_fn in cone_functions]

# Color scale strictly bounded by control authority (vmax = u_max)
vmin = float(np.floor(d_true_map.min() * 10) / 10.0)  # e.g., 0.10
vmax = u_max                                          # e.g., 1.00

# Color levels strictly within the Controllable Trust Region [0.1, 1.0]
levels = np.linspace(vmin, vmax, 20)


# VISUALIZATION: Ground Truth vs. Individual Cones

x1_coords = grid.coordinate_vectors[0]
x2_coords = grid.coordinate_vectors[1]

num_plots = 1 + len(cone_maps)  # Total number of subplots: 1 for ground truth + 4 cones
fig_width = 26.5/5 * num_plots

fig, axes = plt.subplots(1, num_plots, figsize=(fig_width, 6.0), sharey=True)

for ax in axes:
    ax.set_aspect('equal')

# Plot 0: Ground Truth D_true(x)
cf0 = axes[0].contourf(x1_coords, x2_coords, d_true_map.T, levels=levels, cmap='viridis', extend='max')
axes[0].scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', marker='*', s=150, zorder=5)
axes[0].set_title("Ground Truth $D_{true}(x)$", fontweight="bold", fontsize=11)
axes[0].set_xlabel("Position ($x$)")
axes[0].set_ylabel("Velocity ($\\dot{x}$)")
axes[0].grid(True, linestyle=":", alpha=0.5)

# Plots 1-4: Individual Cones showing Controllable Trust Circles
for i, cone_map in enumerate(cone_maps):
    ax = axes[i + 1]
    
    # Fill contours ONLY inside the controllable region (D <= u_max)
    cf = ax.contourf(x1_coords, x2_coords, cone_map.T, levels=levels, cmap='viridis')
    
    # Draw red dotted Trust Boundary (D = u_max)
    ax.contour(x1_coords, x2_coords, cone_map.T, levels=[u_max], colors='red', linestyles='dotted', linewidths=2.0)
    
    # Draw sample marker
    ax.scatter(sampled_points[i, 0], sampled_points[i, 1], color='red', marker='*', s=150, zorder=5)
    
    # Calculate analytical trust radius for title
    d_val = float(sampled_disturbances[i])
    r_trust = max(0.0, (u_max - d_val) / L_D_target) if d_val <= u_max else 0.0
    
    ax.set_title(f"Cone {i+1} ($d_{{i}}={d_val:.2f}, r_{{trust}}={r_trust:.2f}$)", fontweight="bold", fontsize=10)
    ax.set_xlabel("Position ($x$)")
    ax.grid(True, linestyle=":", alpha=0.5)

# Colorbar
cbar_ax = fig.add_axes([0.92, 0.18, 0.01, 0.68])
cbar = fig.colorbar(cf0, cax=cbar_ax, format='%.2f')
cbar.set_label(f"Controllable Disturbance Scale $D(x) ≤ u_{{max}} = {u_max}$", labelpad=15)

plt.subplots_adjust(left=0.04, right=0.91, bottom=0.18, top=0.88, wspace=0.02)
plt.show()

In [ ]:
# PLOT NOMINAL TRAJECTORY AND SAMPLE WAYPOINTS WITH CONES

### Nominal double integrator dynamics

# Initial state and simulation parameters
x_init = jnp.array([-8.0, 3.5])   # Start position = -8.0, start velocity = +3.5
dt = 0.04
total_steps = 200

# Simple PD feedback controller to steer toward origin [0, 0]
def nominal_controller(state):
    x1, x2 = state[0], state[1]
    u_raw = -0.8 * x1 - 1.2 * x2
    return jnp.clip(u_raw, -u_max, u_max)

# Simulate continuous trajectory
traj_states = [x_init]
curr_state = x_init

for _ in range(total_steps):
    u = nominal_controller(curr_state)
    # Double integrator Euler step: x1_new = x1 + x2*dt, x2_new = x2 + u*dt
    next_x1 = curr_state[0] + curr_state[1] * dt
    next_x2 = curr_state[1] + u * dt
    curr_state = jnp.array([next_x1, next_x2])
    traj_states.append(curr_state)

traj_states = jnp.array(traj_states)  # Shape: (201, 2)


# Sample points along the nominal trajectory

num_samples = 10
sample_indices = np.linspace(0, len(traj_states) - 1, num=num_samples, dtype=int)
sampled_points = traj_states[sample_indices]

# Measure ground truth disturbance at each sampled waypoint
sampled_disturbances = jax.vmap(d_true_fn)(sampled_points)

# Evaluate grid maps
d_true_map = jax.vmap(jax.vmap(d_true_fn))(grid.states)
d_combined_map = jax.vmap(jax.vmap(combined_disturbance_fn))(grid.states)


# VISUALIZATION: Ground Truth vs. Combined Landscape with Nominal Trajectory and Sampled Points

vmin = float(np.floor(d_true_map.min() * 10) / 10.0)
vmax = u_max
levels = np.linspace(vmin, vmax, 20)

x1_coords = grid.coordinate_vectors[0]
x2_coords = grid.coordinate_vectors[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6.0), sharey=False)

for ax in axes:
    ax.set_aspect('equal')

# --- Plot 1: Ground Truth D_true(x) with Nominal Trajectory ---
cf0 = axes[0].contourf(x1_coords, x2_coords, d_true_map.T, levels=levels, cmap='viridis', extend='max')

# Dotted black nominal trajectory
axes[0].plot(traj_states[:, 0], traj_states[:, 1], color='black', linestyle=':', linewidth=2.5, label='Nominal Trajectory')
axes[0].scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', marker='o', s=70, zorder=5, label='Sampled Waypoints')

axes[0].set_title("Ground Truth $D_{true}(x)$ with Trajectory", fontweight="bold", fontsize=11)
axes[0].set_xlabel("Position ($x$)")
axes[0].set_ylabel("Velocity ($\\dot{x}$)")
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, linestyle=":", alpha=0.5)

# --- Plot 2: Combined Landscape D_combined(x) ---
cf1 = axes[1].contourf(x1_coords, x2_coords, d_combined_map.T, levels=levels, cmap='viridis')
axes[1].contour(x1_coords, x2_coords, d_combined_map.T, levels=[u_max], colors='red', linestyles='dotted', linewidths=2.0)
axes[1].plot(traj_states[:, 0], traj_states[:, 1], color='black', linestyle=':', linewidth=2.5, label='Nominal Trajectory')
axes[1].scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', marker='*', s=150, zorder=5)

axes[1].set_title("Combined Overapproximation $\\bar{D}_{combined}(x) = \\min_i \\bar{D}_i(x)$", fontweight="bold", fontsize=12)
axes[1].set_xlabel("Position ($x$)")
axes[1].set_ylabel("Velocity ($\\dot{x}$)")
axes[1].grid(True, linestyle=":", alpha=0.5)

# --- Colorbar ---
cbar_ax = fig.add_axes([0.92, 0.18, 0.015, 0.68])
cbar = fig.colorbar(cf0, cax=cbar_ax, format='%.2f')
cbar.set_label(f"Controllable Disturbance Scale $D(x) ≤ u_{{max}} = {u_max}$", labelpad=15, fontsize=11)

plt.subplots_adjust(left=0.08, right=0.90, bottom=0.15, top=0.88, wspace=0.04)
plt.show()

In [ ]:
# Visualization (Poster version)

vmin = float(np.floor(d_true_map.min() * 10) / 10.0)
vmax = u_max
levels = np.linspace(vmin, vmax, 20)

x1_coords = grid.coordinate_vectors[0]
x2_coords = grid.coordinate_vectors[1]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)

for ax in axes:
    ax.set_aspect('equal')
    # Limit number of ticks to unclutter the poster axes
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    # Increase tick number font size
    ax.tick_params(axis='both', which='major', labelsize=12, labelleft=True)


# --- Plot 1: Ground Truth D_true(x) with Nominal Trajectory ---
cf0 = axes[0].contourf(
    x1_coords,
    x2_coords,
    d_true_map.T,
    levels=levels,
    cmap='viridis',
    extend='max',
)

# Dotted black nominal trajectory
axes[0].plot(
    traj_states[:, 0],
    traj_states[:, 1],
    color='black',
    linestyle=':',
    linewidth=3.0,
    label='Nominal Trajectory',
)
axes[0].scatter(
    sampled_points[:, 0],
    sampled_points[:, 1],
    color='red',
    marker='o',
    s=90,
    zorder=5,
    label='Sampled Points',
)

axes[0].set_title(
    'Ground Truth $D_{true}(x)$',
    fontweight='bold',
    fontsize=15,
    pad=10,
)
axes[0].set_xlabel('Position ($x$)', fontsize=14, labelpad=6)
# axes[0].set_ylabel('Velocity ($\\dot{x}$)', fontsize=14, labelpad=6)
axes[0].legend(loc='upper right', fontsize=11, framealpha=0.9)
axes[0].grid(True, linestyle=':', alpha=0.5)

# --- Plot 2: Combined Landscape D_combined(x) ---
cf1 = axes[1].contourf(
    x1_coords, x2_coords, d_combined_map.T, levels=levels, cmap='viridis'
)
axes[1].contour(
    x1_coords,
    x2_coords,
    d_combined_map.T,
    levels=[u_max],
    colors='red',
    linestyles='dotted',
    linewidths=2.5,
)
axes[1].plot(
    traj_states[:, 0],
    traj_states[:, 1],
    color='black',
    linestyle=':',
    linewidth=3.0,
    label='Nominal Trajectory',
)
axes[1].scatter(
    sampled_points[:, 0],
    sampled_points[:, 1],
    color='red',
    marker='*',
    s=180,
    zorder=5,
)

axes[1].set_title(
    'Combined Overapproximation $\\bar{D}_{combined}(x)$',
    fontweight='bold',
    fontsize=15,
    pad=10,
)
axes[1].set_xlabel('Position ($x$)', fontsize=14, labelpad=6)
# axes[1].set_ylabel('Velocity ($\\dot{x}$)', fontsize=14, labelpad=6)
axes[1].grid(True, linestyle=':', alpha=0.5)

# --- Colorbar ---
cbar_ax = fig.add_axes([0.91, 0.16, 0.015, 0.70])
cbar = fig.colorbar(cf0, cax=cbar_ax, format='%.2f')
cbar.set_label(
    f'Controllable Disturbance Scale $D(x) ≤ u_{{max}} = {u_max}$',
    labelpad=15,
    fontsize=13,
)
cbar.ax.tick_params(labelsize=12)

fig.supylabel('Velocity ($\\dot{x}$)', fontsize=14, x=0.02)
plt.subplots_adjust(left=0.04, right=0.90, bottom=0.14, top=0.88, wspace=-0.1)
plt.show()